# Module 3: Deep-Learning Prediction Engine for Bolus Insulin Dosage
### Research Project: *Prediction Of Drug Dosage Of Diabetic Patients Based On Ontology Mapping And Deep Learning Models*

This notebook provides the heavy deep-learning training environment for Google Colab (GPU accelerated) or local execution. It trains and compares:
1. **LSTM Baseline**
2. **GRU Baseline**
3. **BiLSTM Baseline**
4. **Proposed GCN + LSTM Architecture** (Ontology-grounded Knowledge Graph message passing)

**Target:** `target_current_bolus_dose` (Rapid-acting bolus insulin in Units at prediction instant $t$).
**Zero-Leakage Guarantee:** Current bolus at time $t$ is strictly isolated from model input.

## 1. Environment & Setup
Install required dependencies and configure GPU runtime.

In [ ]:
# Install PyTorch Geometric and required packages in Google Colab
!pip install -q torch-geometric pandas numpy scikit-learn matplotlib pyyaml

import os
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
repo_root = os.path.abspath(".")
if repo_root not in sys.path:
    sys.path.append(repo_root)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

## 2. Dataset Loading & Target Distribution
Loads PyTorch Geometric graphs generated by Module 2 (`data/graphs/{train,val,test}.pt`).

In [ ]:
from module3.data.graph_dataset import load_pyg_split, create_dataloader, create_weighted_dataloader

DATA_DIR = "data/graphs"
train_graphs = load_pyg_split(DATA_DIR, "train")
val_graphs = load_pyg_split(DATA_DIR, "val")
test_graphs = load_pyg_split(DATA_DIR, "test")

train_y = np.array([float(g.y.item()) for g in train_graphs])
val_y = np.array([float(g.y.item()) for g in val_graphs])
test_y = np.array([float(g.y.item()) for g in test_graphs])

print("=" * 60)
print(f"TRAIN GRAPHS: {len(train_graphs):,} | Active Bolus (>0): {(train_y > 0.1).sum():,} ({(train_y > 0.1).mean()*100:.2f}%)")
print(f"VAL GRAPHS:   {len(val_graphs):,} | Active Bolus (>0): {(val_y > 0.1).sum():,} ({(val_y > 0.1).mean()*100:.2f}%)")
print(f"TEST GRAPHS:  {len(test_graphs):,} | Active Bolus (>0): {(test_y > 0.1).sum():,} ({(test_y > 0.1).mean()*100:.2f}%)")
print("=" * 60)

# Visualize target imbalance distribution
plt.figure(figsize=(8, 3.5))
plt.hist(train_y[train_y > 0.1], bins=20, color="#2563eb", edgecolor="black", alpha=0.8)
plt.title("Distribution of Non-Zero Active Bolus Targets (y > 0.1 U)")
plt.xlabel("Bolus Dose (Units)")
plt.ylabel("Frequency")
plt.grid(alpha=0.3)
plt.show()

## 3. Model Configuration & Selection
Configure model architecture, weighted Huber loss, and non-negative Softplus activation.

In [ ]:
from module3.config import ModelConfig, TrainConfig
from module3.models import build_model

# Select architecture: 'gcn_lstm', 'lstm', 'gru', or 'bilstm'
MODEL_TYPE = "gcn_lstm"

model_config = ModelConfig(
    model_type=MODEL_TYPE,
    in_channels=15,
    gcn_hidden_dim=64,
    temporal_hidden_dim=64,
    dropout=0.2,
)

train_config = TrainConfig(
    batch_size=16,
    epochs=10,
    learning_rate=1e-3,
    loss_fn="weighted_smooth_l1",
    pos_weight=8.0,  # 8.0x loss penalty on meal bolus samples to prevent zero-collapse
    early_stopping_patience=5,
    seed=42,
    device=str(device),
)

model = build_model(model_config)
print(f"Initialized {MODEL_TYPE.upper()} with {model.count_parameters():,} trainable parameters.")

## 4. Model Training with Early Stopping & Checkpointing

In [ ]:
from module3.training.trainer import Trainer

train_loader = create_weighted_dataloader(
    train_graphs,
    batch_size=train_config.batch_size,
    pos_weight=train_config.pos_weight,
    shuffle=True,
)
val_loader = create_dataloader(val_graphs, batch_size=train_config.batch_size, shuffle=False)
test_loader = create_dataloader(test_graphs, batch_size=train_config.batch_size, shuffle=False)

trainer = Trainer(
    model=model,
    config=train_config,
    model_name=MODEL_TYPE,
    save_dir="data/models",
)

train_history = trainer.fit(train_loader, val_loader)

## 5. Training & Validation Loss Curves

In [ ]:
hist = train_history["history"]
epochs_range = hist["epoch"]

plt.figure(figsize=(10, 4))
plt.plot(epochs_range, hist["train_loss"], label="Train Loss (Weighted Huber)", marker="o", color="#2563eb")
plt.plot(epochs_range, hist["val_loss"], label="Validation Loss", marker="s", color="#ef4444")
plt.title(f"{MODEL_TYPE.upper()} Training & Validation Loss Curves")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 6. Test Set Evaluation & Prediction Analysis
Evaluate the best saved checkpoint on the held-out test split.

In [ ]:
from module3.evaluation.evaluator import Evaluator
from module3.config import ExperimentConfig

exp_config = ExperimentConfig(model=model_config, train=train_config)
evaluator = Evaluator(exp_config)

results = evaluator.evaluate_model(
    model=model,
    test_loader=test_loader,
    model_name=MODEL_TYPE,
    checkpoint_path=train_history["checkpoint_path"],
)

# Load and inspect individual predictions
pred_df = pd.read_csv(results["predictions_csv"])
print(pred_df[pred_df["actual_bolus_dose"] > 0.1].head(10))

# Plot Actual vs Predicted Bolus Scatter
plt.figure(figsize=(6, 6))
plt.scatter(pred_df["actual_bolus_dose"], pred_df["predicted_bolus_dose"], alpha=0.6, color="#10b981", edgecolors="black")
max_val = max(pred_df["actual_bolus_dose"].max(), pred_df["predicted_bolus_dose"].max()) + 1
plt.plot([0, max_val], [0, max_val], "r--", label="Perfect Prediction")
plt.title(f"{MODEL_TYPE.upper()}: Actual vs Predicted Bolus Dosage (Units)")
plt.xlabel("Actual Bolus (Units)")
plt.ylabel("Predicted Bolus (Units)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()